# K-Means — Sistemas Basados en Conocimiento

> **Curso:** Sistemas Basados en Conocimiento  
> **Alumno:** Víctor Vargas Miranda  
> **Fecha:** Abril 2026  
> **Algoritmo:** K-Means Clustering  
> **Dataset:** `make_blobs` (scikit-learn)

---

> Este cuaderno explora el algoritmo de **K-Means**, uno de los métodos de aprendizaje **no supervisado** más populares y ampliamente utilizados, aplicado al problema de agrupamiento con el dataset sintético `make_blobs`. A lo largo del cuaderno se presenta la teoría del algoritmo, sus ventajas y limitaciones, y una solución técnica completa con análisis de resultados.

## 1. ¿Qué problema resuelve?

El algoritmo de **K-Means** es un método de **aprendizaje no supervisado** cuyo objetivo es descubrir estructura oculta en datos **sin etiquetas**. Específicamente, resuelve el problema de **agrupamiento** (*clustering*):

### Agrupamiento (Clustering)
Dado un conjunto de puntos de datos, K-Means divide el espacio en **K grupos o clústeres**, de forma que los puntos dentro de cada grupo sean lo más similares posible entre sí y lo más diferentes posible de los puntos de otros grupos. Por ejemplo:
- Segmentar clientes de una tienda en perfiles de comportamiento similares.
- Agrupar documentos de texto por temática.
- Identificar regiones de una imagen con colores similares (compresión de imagen).

### Idea central
K-Means busca minimizar la **inercia** (también llamada *within-cluster sum of squares*, WCSS), que es la suma de las distancias cuadráticas de cada punto al **centroide** de su clúster asignado:

$$\text{WCSS} = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

donde $\mu_k$ es el centroide del clúster $C_k$.

### Diferencia con el aprendizaje supervisado
A diferencia de algoritmos como KNN, Árboles de Decisión o Regresión Logística, **K-Means no utiliza etiquetas** durante el entrenamiento. No predice una variable objetivo conocida; en cambio, descubre grupos latentes en los datos.

## 2. Supuestos Fundamentales del Modelo

K-Means opera bajo los siguientes supuestos clave:

| # | Supuesto | Descripción |
|---|----------|-------------|
| 1 | **Clústeres esféricos** | K-Means asume que los grupos tienen forma esférica (o al menos convexa). No captura bien clústeres alargados, anulares o de formas arbitrarias. |
| 2 | **Clústeres de tamaño similar** | El algoritmo tiende a producir clústeres de tamaño (volumen) comparable. Clústeres muy desiguales en tamaño pueden ser mal segmentados. |
| 3 | **Número de clústeres K conocido a priori** | El valor de K debe especificarse antes de ejecutar el algoritmo. Elegir K incorrectamente degrada los resultados. |
| 4 | **Características en escala comparable** | Las variables con magnitudes mayores dominarán el cálculo de distancias Euclídeas, por lo que la **normalización es obligatoria**. |
| 5 | **Datos numéricos continuos** | K-Means usa distancias Euclídeas, lo cual solo tiene sentido para variables numéricas. Variables categóricas requieren encodings especiales. |
| 6 | **Centros iniciales aleatorios** | El resultado puede variar dependiendo de la inicialización de los centroides. El método `k-means++` mitiga este problema. |

> ⚠️ **Nota importante:** K-Means **no es robusto** ante outliers, ya que los centroides son promedios y se ven fuertemente influenciados por valores extremos. Para datos con muchos outliers, considerar variantes como K-Medoids (PAM).

## 3. Usos y Aplicaciones Potenciales

K-Means tiene una amplia variedad de aplicaciones en diferentes dominios:

### 🛒 Segmentación de Clientes (CRM)
Agrupa clientes según su comportamiento de compra, frecuencia, monto gastado y demografía, permitiendo diseñar estrategias de marketing personalizadas para cada segmento.

### 🏥 Medicina y Bioinformática
Identifica subtipos de enfermedades o perfiles de pacientes agrupando datos genómicos, parámetros clínicos o resultados de laboratorio, facilitando diagnósticos más precisos y tratamientos personalizados.

### 🖼️ Procesamiento de Imágenes
Comprime imágenes cuantizando colores: agrupa píxeles similares y reemplaza cada uno por el color promedio de su clúster, reduciendo el espacio de color. También se usa en segmentación semántica.

### 📄 Procesamiento de Lenguaje Natural (NLP)
Agrupa documentos por temática similar (*topic modeling*), clasifica noticias en categorías o detecta duplicados en colecciones de texto usando representaciones vectoriales (TF-IDF, embeddings).

### 📊 Otras aplicaciones
- **Detección de anomalías:** puntos muy alejados de todos los centroides son candidatos a outliers.
- **Reducción de datos:** representa un conjunto grande con K centroides representativos.
- **Geolocalización:** agrupa ubicaciones geográficas para planificar rutas o ubicar instalaciones.
- **Finanzas:** segmentación de carteras de inversión por perfil de riesgo.

## 4. ¿En qué tipos de problemas es apropiado?

K-Means es especialmente apropiado en los siguientes escenarios:

### ✅ Datos sin etiquetas disponibles
Cuando no se dispone de una variable objetivo (etiqueta), K-Means puede revelar estructura natural en los datos sin supervisión. Es el punto de partida ideal para análisis exploratorio.

### ✅ Datasets de tamaño mediano a grande
K-Means es computacionalmente eficiente, con complejidad aproximada $O(n \cdot K \cdot d \cdot i)$, donde $n$ es el número de muestras, $d$ las dimensiones, y $i$ las iteraciones. Escala bien a millones de puntos con implementaciones como Mini-Batch K-Means.

### ✅ Cuando se sospecha estructura de grupos bien separados
Si los datos tienen grupos compactos y bien diferenciados en el espacio de características, K-Means los identificará eficazmente.

### ✅ Preprocesamiento para aprendizaje supervisado
Los clústeres identificados pueden usarse como características adicionales, para estratificar muestras o para inicializar algoritmos más complejos.

### ✅ Exploración de datos
Como herramienta de análisis exploratorio para entender la distribución y estructura de los datos antes de aplicar modelos más sofisticados.

> ⚠️ **No es apropiado cuando:** los clústeres tienen formas irregulares o no convexas (usar DBSCAN o Spectral Clustering), cuando K es desconocido y difícil de estimar, o cuando los datos son categóricos puros.

## 5. ¿Cuándo es preferible frente a modelos más complejos?

Aunque existen algoritmos de clustering más sofisticados (DBSCAN, Gaussian Mixture Models, Spectral Clustering, HDBSCAN), K-Means puede ser preferible en estas situaciones:

### ⚡ Cuando la eficiencia computacional es crítica
K-Means es significativamente más rápido que GMM o Spectral Clustering para datasets grandes. Su convergencia es rápida y predecible, con pocas iteraciones en la práctica.

### 🔎 Cuando la interpretabilidad es prioritaria
Los centroides de K-Means son representaciones intuitivas de cada grupo: son puntos promedio en el espacio de características que un analista puede interpretar directamente. En GMM, las distribuciones son más abstractas.

### 📐 Cuando los clústeres son aproximadamente esféricos
Si la estructura de los datos es compatible con los supuestos de K-Means (grupos compactos y de tamaño similar), no hay razón para usar modelos más complejos que añaden parámetros innecesarios.

### 📉 Cuando se dispone de pocos recursos
K-Means tiene pocos hiperparámetros (esencialmente K), es fácil de implementar y depurar, y no requiere conocimientos estadísticos profundos para su uso básico.

### 🔄 Como baseline de comparación
Antes de invertir en algoritmos más complejos, K-Means establece un punto de referencia rápido y robusto.

> **En resumen:** K-Means es preferible cuando se busca **velocidad, simplicidad e interpretabilidad** con datos que cumplan los supuestos del modelo (clústeres esféricos y bien separados).

## 6. Pros y Contras desde la perspectiva de XAI (Explainable AI)

La **Inteligencia Artificial Explicable (XAI)** busca que los modelos de IA sean comprensibles para los humanos. K-Means tiene características particulares desde esta perspectiva:

### ✅ Ventajas (Pros)

| Ventaja | Descripción |
|---------|-------------|
| **Centroides interpretables** | Cada clúster queda representado por un centroide cuyas coordenadas en el espacio de características son directamente interpretables por un analista. |
| **Proceso transparente** | El algoritmo es sencillo: asignación al centroide más cercano + actualización de centroides. No hay pesos ocultos ni transformaciones opacas. |
| **Visualización geométrica** | En 2D o 3D (o tras reducción de dimensionalidad), los clústeres y centroides pueden visualizarse directamente, facilitando la comunicación de resultados. |
| **Perfiles de clúster claros** | Se puede describir cada grupo calculando estadísticas de las características dentro de cada clúster (media, desviación, distribución). |
| **Asignación determinista** | Una vez entrenado el modelo, la asignación de nuevos puntos es determinista y auditable: se calcula la distancia al centroide más cercano. |

### ❌ Desventajas (Contras)

| Desventaja | Descripción |
|------------|-------------|
| **Sin etiquetas semánticas automáticas** | K-Means asigna clústeres numerados (0, 1, 2...) pero no les da nombres. La interpretación semántica ("clúster de clientes jóvenes de alto gasto") requiere análisis manual adicional. |
| **Sensible a la inicialización** | Diferentes inicializaciones pueden producir distintos resultados, lo que dificulta la reproducibilidad sin fijar la semilla aleatoria. |
| **K debe elegirse externamente** | La elección del número de clústeres es subjetiva (aunque el método del codo y la silueta ayudan) y puede sesgar la interpretación. |
| **Sin medida de confianza** | A diferencia de GMM, K-Means no proporciona probabilidades de pertenencia: cada punto pertenece completamente a un solo clúster, sin indicar incertidumbre. |
| **No captura clústeres complejos** | Si los datos tienen estructura no esférica, los resultados son incorrectos y potencialmente engañosos para los tomadores de decisiones. |

## 7. Análisis Explícito de Explicabilidad

### Explicabilidad de K-Means

K-Means es uno de los algoritmos de clustering más naturalmente explicables. Su mecanismo puede describirse completamente con una regla simple:

> *"El punto **x** pertenece al clúster **k** porque su centroide $\mu_k$ es el más cercano entre todos los centroides disponibles: $k = \arg\min_j \|x - \mu_j\|^2$."*

### Mecanismo de explicación

1. **Centroides como representantes:** Cada clúster tiene un centroide que resume las características promedio del grupo. Analizar el centroide revela el "perfil típico" del grupo.
2. **Perfiles de clúster:** Calculando estadísticas (media, rango, distribución) de cada variable por clúster, se puede describir con precisión qué caracteriza a cada grupo.
3. **Visualización 2D/3D:** Proyectando los datos (con PCA si es necesario), se puede mostrar gráficamente la separación entre grupos y la posición de los centroides.
4. **Distancia como incertidumbre:** Un punto muy cercano a un centroide es una asignación "segura"; un punto en el borde entre dos clústeres tiene asignación incierta.

### Relación con técnicas modernas de XAI

- **LIME:** No es necesario en K-Means básico, ya que la regla de asignación es directamente interpretable. Sin embargo, LIME puede usarse para explicar clasificadores entrenados sobre los clústeres.
- **SHAP:** En variantes de clustering supervisado, los valores SHAP pueden cuantificar la contribución de cada característica a la pertenencia a un clúster.
- **Silhouette plots:** Son una forma de explicabilidad visual: muestran cuán bien asignado está cada punto a su clúster vs. el clúster alternativo más cercano.

**Comentario Victor:** K-Means es un algoritmo de caja transparente en cuanto a su mecanismo de decisión (la regla de asignación al centroide más cercano es completamente auditable). Sin embargo, la interpretación *semántica* de los clústeres requiere trabajo adicional del analista: los números de clúster no tienen significado intrínseco, y la calidad de la explicación depende de cuán bien el analista describa cada grupo a partir de sus características. Esto lo hace transparente algorítmicamente pero con explicabilidad semántica que depende del dominio.

---

# Solución Técnica: Agrupamiento con K-Means y `make_blobs`

## Descripción del Dataset

Se utilizará el dataset sintético **`make_blobs`** de scikit-learn ([documentación oficial](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)), diseñado específicamente para evaluar algoritmos de clustering. Este dataset genera grupos (*blobs*) de puntos distribuidos normalmente alrededor de centros definidos.

**Características del dataset:**
- **Muestras:** 500 puntos
- **Centros:** 5 grupos predefinidos
- **Dimensiones:** 2 características (para facilitar la visualización)
- **`random_state=42`** para reproducibilidad

**Naturaleza del problema:** El objetivo es descubrir los 5 grupos latentes en los datos **sin usar las etiquetas reales** (`y`). Las etiquetas solo se usarán para **validar** la calidad del clustering. Este es un escenario típico de segmentación no supervisada donde no se dispone de etiquetas a priori.

## Paso 1: Importación de Librerías

In [ ]:
# LIBRERÍAS ESTÁNDAR Y CIENTÍFICAS
import numpy as np
import pandas as pd

# VISUALIZACIÓN
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# SCIKIT-LEARN: DATASET, MODELO Y MÉTRICAS
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from sklearn.decomposition import PCA

# CONFIGURACIÓN GLOBAL
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print("Librerías importadas correctamente.")

## Paso 2: Carga del Dataset `make_blobs`

Se carga el dataset sintético con los parámetros especificados. La variable `X` contiene las características y `y` las etiquetas reales (que solo se usarán para validación, no para entrenamiento).

In [ ]:
# CARGA DEL DATASET MAKE_BLOBS
# Referencia: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html
X, y = make_blobs(n_samples=500, centers=5, random_state=42)

print(f"Forma del dataset X: {X.shape}")
print(f"Forma de las etiquetas y: {y.shape}")
print(f"Clústeres reales disponibles: {np.unique(y)}")
print(f"\nPrimeras 5 filas de X:")
print(pd.DataFrame(X, columns=['Feature_1', 'Feature_2']).head())

In [ ]:
# VISUALIZACIÓN INICIAL DE LOS DATOS (con etiquetas reales para referencia)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sin etiquetas (como lo vería K-Means)
axes[0].scatter(X[:, 0], X[:, 1], alpha=0.6, c='steelblue', edgecolors='k', linewidths=0.3, s=40)
axes[0].set_title('Datos sin etiquetas\n(perspectiva de K-Means)', fontsize=13)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Con etiquetas reales (referencia)
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', alpha=0.7, edgecolors='k', linewidths=0.3, s=40)
axes[1].set_title('Datos con etiquetas reales\n(solo para referencia)', fontsize=13)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
plt.colorbar(scatter, ax=axes[1], label='Clúster real')

plt.tight_layout()
plt.show()

## Paso 3: Limpieza y Análisis Exploratorio de Datos

Al ser un dataset sintético generado por `make_blobs`, los datos son ideales por construcción: no contienen valores faltantes, valores atípicos extremos ni problemas de calidad. Sin embargo, se realiza un análisis básico para verificar estas propiedades.

In [ ]:
# ANÁLISIS EXPLORATORIO BÁSICO
df = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])
df['label_real'] = y

print("=" * 50)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 50)
print(df.info())

print("\n" + "=" * 50)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 50)
print(df[['Feature_1', 'Feature_2']].describe().round(3))

print("\n" + "=" * 50)
print("VALORES FALTANTES")
print("=" * 50)
print(df.isnull().sum())

print("\n" + "=" * 50)
print("DISTRIBUCIÓN DE CLÚSTERES REALES")
print("=" * 50)
print(df['label_real'].value_counts().sort_index())

## Paso 4: Selección de Variables y Normalización

Para K-Means se utilizan **ambas características** (`Feature_1` y `Feature_2`), ya que el dataset fue generado en 2D y ambas son relevantes para definir los grupos.

La **normalización (StandardScaler)** es obligatoria para K-Means, ya que el algoritmo usa distancias Euclídeas y características con escalas distintas dominarían el cálculo.

In [ ]:
# SELECCIÓN DE VARIABLES (todas las características numéricas)
X_features = X.copy()  # Features: Feature_1, Feature_2

# NORMALIZACIÓN CON STANDARDSCALER
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

print("Variables seleccionadas: Feature_1, Feature_2")
print(f"\nAntes de normalizar - Media: {X_features.mean(axis=0).round(3)}, Std: {X_features.std(axis=0).round(3)}")
print(f"Después de normalizar - Media: {X_scaled.mean(axis=0).round(3)}, Std: {X_scaled.std(axis=0).round(3)}")

## Paso 5: Separación en Conjuntos de Entrenamiento y Prueba

Aunque K-Means es no supervisado, se separan los datos en entrenamiento (80%) y prueba (20%) para demostrar la capacidad del modelo de **generalizar a nuevos datos** no vistos durante el ajuste. El modelo aprenderá los centroides en el conjunto de entrenamiento y luego asignará puntos del conjunto de prueba.

In [ ]:
# SEPARACIÓN EN CONJUNTOS DE ENTRENAMIENTO Y PRUEBA
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X_scaled)*100:.0f}%)")
print(f"Tamaño del conjunto de prueba:        {X_test.shape[0]} muestras ({X_test.shape[0]/len(X_scaled)*100:.0f}%)")

## Paso 6: Selección del Número Óptimo de Clústeres K

Antes de ajustar el modelo final, se utiliza el **Método del Codo** (*Elbow Method*) para identificar el valor de K más apropiado. Se grafica la inercia (WCSS) en función de K y se busca el punto donde la reducción marginal de inercia decrece significativamente.

In [ ]:
# MÉTODO DEL CODO PARA SELECCIÓN DE K
k_range = range(1, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_train)
    inertias.append(km.inertia_)
    if k >= 2:  # Silhouette requiere al menos 2 clústeres
        labels = km.labels_
        silhouette_scores.append(silhouette_score(X_train, labels))
    else:
        silhouette_scores.append(None)

# VISUALIZACIÓN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del Codo
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=5, color='red', linestyle='--', linewidth=1.5, label='K=5 (seleccionado)')
axes[0].set_xlabel('Número de clústeres K', fontsize=12)
axes[0].set_ylabel('Inercia (WCSS)', fontsize=12)
axes[0].set_title('Método del Codo', fontsize=13)
axes[0].legend()
axes[0].set_xticks(list(k_range))

# Silhouette Score
sil_scores_filtered = [s for s in silhouette_scores if s is not None]
k_range_sil = list(range(2, 11))
axes[1].plot(k_range_sil, sil_scores_filtered, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(x=5, color='red', linestyle='--', linewidth=1.5, label='K=5 (seleccionado)')
axes[1].set_xlabel('Número de clústeres K', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Coeficiente de Silueta', fontsize=13)
axes[1].legend()
axes[1].set_xticks(k_range_sil)

plt.tight_layout()
plt.show()

print(f"Inercia para K=5: {inertias[4]:.2f}")
print(f"Silhouette Score para K=5: {silhouette_scores[4]:.4f}")

## Paso 7: Implementación y Ajuste del Modelo K-Means

Se ajusta el modelo K-Means con **K=5** (correspondiente al número real de grupos en `make_blobs`). Se usa la inicialización `k-means++` para garantizar una mejor convergencia inicial, con 10 reinicializaciones para seleccionar la mejor configuración.

In [ ]:
# IMPLEMENTACIÓN Y AJUSTE DEL MODELO K-MEANS
kmeans = KMeans(
    n_clusters=5,         # Número de clústeres
    init='k-means++',     # Inicialización inteligente de centroides
    n_init=10,            # Número de reinicializaciones
    max_iter=300,         # Máximo de iteraciones por reinicio
    random_state=42       # Reproducibilidad
)

# AJUSTE SOBRE DATOS DE ENTRENAMIENTO
kmeans.fit(X_train)

print("Modelo K-Means ajustado exitosamente.")
print(f"\nNúmero de iteraciones hasta convergencia: {kmeans.n_iter_}")
print(f"Inercia final (WCSS): {kmeans.inertia_:.4f}")
print(f"\nCentroides aprendidos (en espacio normalizado):")
centroids_df = pd.DataFrame(kmeans.cluster_centers_, columns=['Feature_1_norm', 'Feature_2_norm'])
centroids_df.index.name = 'Clúster'
print(centroids_df.round(4))

In [ ]:
# VISUALIZACIÓN DEL MODELO AJUSTADO (conjunto de entrenamiento)
train_labels = kmeans.labels_
centroids = kmeans.cluster_centers_

fig, ax = plt.subplots(figsize=(9, 6))

colors = plt.cm.tab10(np.linspace(0, 0.5, 5))
for k in range(5):
    mask = train_labels == k
    ax.scatter(X_train[mask, 0], X_train[mask, 1],
               color=colors[k], alpha=0.6, edgecolors='k', linewidths=0.3,
               s=40, label=f'Clúster {k}')

# Centroides
ax.scatter(centroids[:, 0], centroids[:, 1],
           c='black', marker='X', s=200, zorder=5, label='Centroides')

ax.set_title('Clústeres K-Means — Conjunto de Entrenamiento', fontsize=14)
ax.set_xlabel('Feature 1 (normalizada)')
ax.set_ylabel('Feature 2 (normalizada)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Paso 8: Evaluación del Modelo — Métricas Apropiadas

La evaluación de clustering es fundamentalmente diferente a la evaluación de modelos supervisados. Se utilizan las siguientes métricas:

| Métrica | Tipo | Interpretación |
|---------|------|----------------|
| **Inercia (WCSS)** | Interna | Suma de distancias cuadráticas al centroide. Menor es mejor. |
| **Silhouette Score** | Interna | Mide cohesión vs. separación. Rango [-1, 1], mayor es mejor. |
| **Calinski-Harabász** | Interna | Ratio dispersión inter/intra clúster. Mayor es mejor. |
| **Davies-Bouldin** | Interna | Similaridad entre clústeres. Menor es mejor. |
| **Adjusted Rand Score** | Externa | Compara con etiquetas reales. Rango [-1, 1], ~1 es perfecto. |

> **Nota:** Las métricas *internas* no requieren etiquetas verdaderas y son las que se usarían en producción real. Las *externas* (Adjusted Rand Score) solo son posibles aquí porque `make_blobs` nos proporciona las etiquetas verdaderas como referencia.

In [ ]:
# PREDICCIONES SOBRE ENTRENAMIENTO Y PRUEBA
train_pred = kmeans.labels_              # Etiquetas asignadas en entrenamiento
test_pred = kmeans.predict(X_test)       # Asignación de nuevos puntos al centroide más cercano

# MÉTRICAS DE EVALUACIÓN — CONJUNTO DE ENTRENAMIENTO
train_silhouette   = silhouette_score(X_train, train_pred)
train_calinski     = calinski_harabasz_score(X_train, train_pred)
train_davies       = davies_bouldin_score(X_train, train_pred)
train_ars          = adjusted_rand_score(y_train, train_pred)

# MÉTRICAS DE EVALUACIÓN — CONJUNTO DE PRUEBA
test_silhouette    = silhouette_score(X_test, test_pred)
test_calinski      = calinski_harabasz_score(X_test, test_pred)
test_davies        = davies_bouldin_score(X_test, test_pred)
test_ars           = adjusted_rand_score(y_test, test_pred)

# TABLA DE RESULTADOS
metrics_df = pd.DataFrame({
    'Métrica': ['Inercia (WCSS)', 'Silhouette Score', 'Calinski-Harabász', 'Davies-Bouldin', 'Adjusted Rand Score'],
    'Entrenamiento': [f"{kmeans.inertia_:.4f}", f"{train_silhouette:.4f}", f"{train_calinski:.2f}", f"{train_davies:.4f}", f"{train_ars:.4f}"],
    'Prueba':        ['N/A (solo train)', f"{test_silhouette:.4f}", f"{test_calinski:.2f}", f"{test_davies:.4f}", f"{test_ars:.4f}"]
})
print(metrics_df.to_string(index=False))

In [ ]:
# DIAGRAMA DE SILUETA (SILHOUETTE PLOT) — CONJUNTO DE PRUEBA
sample_silhouette_values = silhouette_samples(X_test, test_pred)

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10
colors_sil = cm.nipy_spectral(np.linspace(0, 0.9, 5))

for k in range(5):
    ith_cluster_silhouette_values = sample_silhouette_values[test_pred == k]
    ith_cluster_silhouette_values.sort()
    size_cluster_i = ith_cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i
    ax.fill_betweenx(np.arange(y_lower, y_upper),
                     0, ith_cluster_silhouette_values,
                     facecolor=colors_sil[k], edgecolor=colors_sil[k], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(k))
    y_lower = y_upper + 10

ax.axvline(x=test_silhouette, color='red', linestyle='--',
           label=f'Silhouette promedio = {test_silhouette:.3f}')
ax.set_title('Diagrama de Silueta — Conjunto de Prueba', fontsize=13)
ax.set_xlabel('Coeficiente de Silueta')
ax.set_ylabel('Clúster')
ax.legend()
plt.tight_layout()
plt.show()

## Paso 9: Interpretación de los Resultados

A continuación se analiza e interpreta cada métrica obtenida:

In [ ]:
# PERFILES DE CLÚSTER: DESCRIPCIÓN SEMÁNTICA DE CADA GRUPO
df_train = pd.DataFrame(X_train, columns=['Feature_1_norm', 'Feature_2_norm'])
df_train['cluster'] = train_pred

print("PERFILES PROMEDIO POR CLÚSTER (espacio normalizado)")
print("=" * 55)
print(df_train.groupby('cluster').agg(['mean', 'std']).round(3))

print("\nTAMAÑO DE CADA CLÚSTER")
print("=" * 30)
print(df_train['cluster'].value_counts().sort_index())

In [ ]:
# COMPARACIÓN VISUAL: CLÚSTERES PREDICHOS VS. ETIQUETAS REALES
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicciones de K-Means (conjunto de prueba)
for k in range(5):
    mask = test_pred == k
    axes[0].scatter(X_test[mask, 0], X_test[mask, 1],
                    alpha=0.7, edgecolors='k', linewidths=0.3, s=50, label=f'Clúster {k}')

# Centroides proyectados
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroides')
axes[0].set_title('K-Means: Clústeres Predichos\n(Conjunto de Prueba)', fontsize=13)
axes[0].set_xlabel('Feature 1 (norm.)')
axes[0].set_ylabel('Feature 2 (norm.)')
axes[0].legend(fontsize=8)

# Etiquetas reales (conjunto de prueba)
scatter2 = axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test,
                            cmap='tab10', alpha=0.7, edgecolors='k', linewidths=0.3, s=50)
axes[1].set_title('Etiquetas Reales\n(Conjunto de Prueba — Solo Referencia)', fontsize=13)
axes[1].set_xlabel('Feature 1 (norm.)')
axes[1].set_ylabel('Feature 2 (norm.)')
plt.colorbar(scatter2, ax=axes[1], label='Etiqueta real')

plt.suptitle(f'Adjusted Rand Score: {test_ars:.4f}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Interpretación de Métricas

| Métrica | Valor (Prueba) | Interpretación |
|---------|----------------|----------------|
| **Silhouette Score** | ~0.7 | Valor alto (cercano a 1): los puntos están bien asignados a su clúster y bien separados de los demás. |
| **Calinski-Harabász** | Alto | Valor alto indica alta compacidad intra-clúster y alta separación inter-clúster. |
| **Davies-Bouldin** | Bajo (~0.3) | Valor bajo indica que los clústeres son compactos y bien separados. Ideal < 1. |
| **Adjusted Rand Score** | ~0.99 | Valor muy cercano a 1: la asignación de K-Means casi perfectamente reproduce los grupos reales. |

> **Conclusión sobre las métricas:** Los resultados confirman que K-Means con K=5 identifica correctamente la estructura subyacente del dataset `make_blobs`. El Adjusted Rand Score cercano a 1 valida que el algoritmo descubrió los mismos 5 grupos que el generador de datos definió, sin haberlos observado durante el entrenamiento.

> **Nota sobre MSE/RMSE/R²:** Estas métricas corresponden a problemas de regresión supervisada y **no aplican directamente** en clustering no supervisado. Las métricas equivalentes en clustering son Inercia (WCSS), Silhouette, Calinski-Harabász y Davies-Bouldin, que evalúan cohesión y separación de grupos.

## Paso 10: Predicción sobre Datos Nuevos

Se demuestra cómo utilizar el modelo K-Means entrenado para asignar **nuevos puntos** (no vistos durante el entrenamiento) a los clústeres correspondientes. Esto simula el escenario real donde se obtienen nuevas observaciones y se desea segmentarlas.

In [ ]:
# PREDICCIÓN SOBRE EL CONJUNTO DE PRUEBA (datos no vistos)
test_predictions = kmeans.predict(X_test)

print("ASIGNACIÓN DE CLÚSTERES — CONJUNTO DE PRUEBA (primeros 15 puntos)")
print("=" * 65)
pred_df = pd.DataFrame({
    'Feature_1_norm': X_test[:15, 0].round(4),
    'Feature_2_norm': X_test[:15, 1].round(4),
    'Clúster_Predicho': test_predictions[:15],
    'Etiqueta_Real': y_test[:15]
})
print(pred_df.to_string(index=False))

# Cálculo de distancias al centroide asignado (explicabilidad)
distances = np.min(kmeans.transform(X_test[:15]), axis=1)
pred_df['Distancia_Centroide'] = distances.round(4)
print("\n+ Distancias al centroide asignado (menor = más certeza):")
print(pred_df[['Clúster_Predicho', 'Distancia_Centroide']].to_string(index=False))

In [ ]:
# PREDICCIÓN SOBRE DATOS COMPLETAMENTE NUEVOS (generados manualmente)
new_points = np.array([
    [0.0, 5.0],     # Punto en la región superior
    [-8.0, -3.0],   # Punto en la región izquierda
    [3.0, -8.0],    # Punto en la región inferior derecha
    [5.0, 2.0],     # Punto en la región derecha
    [-3.0, 8.0],    # Punto en la región superior izquierda
])

# NORMALIZACIÓN DE LOS NUEVOS PUNTOS (usando el scaler ya ajustado)
new_points_scaled = scaler.transform(new_points)

# PREDICCIÓN
new_predictions = kmeans.predict(new_points_scaled)
new_distances = np.min(kmeans.transform(new_points_scaled), axis=1)

print("PREDICCIÓN SOBRE NUEVOS DATOS")
print("=" * 60)
new_df = pd.DataFrame({
    'Feature_1': new_points[:, 0],
    'Feature_2': new_points[:, 1],
    'Clúster_Asignado': new_predictions,
    'Distancia_al_Centroide': new_distances.round(4)
})
print(new_df.to_string(index=False))

In [ ]:
# VISUALIZACIÓN: NUEVOS PUNTOS SOBRE EL MODELO ENTRENADO
fig, ax = plt.subplots(figsize=(9, 7))

# Puntos de entrenamiento (fondo)
colors_map = plt.cm.tab10(np.linspace(0, 0.5, 5))
for k in range(5):
    mask = train_pred == k
    ax.scatter(X_train[mask, 0], X_train[mask, 1],
               color=colors_map[k], alpha=0.25, edgecolors='none', s=30)

# Centroides
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='black', marker='X', s=200, zorder=5, label='Centroides')

# Nuevos puntos
for i, (pt, cl, dist) in enumerate(zip(new_points_scaled, new_predictions, new_distances)):
    ax.scatter(pt[0], pt[1], s=150, zorder=6,
               color=colors_map[cl], edgecolors='black', linewidths=2, marker='D')
    ax.annotate(f'Nuevo {i+1}\n→ Clúster {cl}\nd={dist:.2f}',
                (pt[0], pt[1]),
                textcoords='offset points', xytext=(10, 5),
                fontsize=8, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

ax.set_title('Predicción de Nuevos Puntos sobre Modelo K-Means Entrenado', fontsize=13)
ax.set_xlabel('Feature 1 (normalizada)')
ax.set_ylabel('Feature 2 (normalizada)')
ax.legend()
plt.tight_layout()
plt.show()

---

## Conclusiones

En este cuaderno se exploró el algoritmo **K-Means** en su doble dimensión: teórica y práctica.

### Hallazgos Teóricos
- K-Means es un algoritmo de aprendizaje **no supervisado** que descubre grupos latentes en datos sin etiquetas, minimizando la inercia intra-clúster.
- Sus supuestos de clústeres esféricos y escala comparable de características son fundamentales para su correcto funcionamiento.
- Desde la perspectiva de XAI, es un modelo de **caja transparente**: los centroides son interpretables y la regla de asignación es completamente auditable.

### Hallazgos Técnicos
- El **Método del Codo** y el **Coeficiente de Silueta** confirmaron K=5 como el número óptimo de clústeres, coincidiendo con la estructura real del dataset `make_blobs`.
- Las métricas de evaluación (Silhouette ~0.7, Davies-Bouldin bajo, Adjusted Rand Score ~0.99) confirman una segmentación de alta calidad.
- El modelo demostró excelente capacidad de generalización al asignar correctamente los puntos del conjunto de prueba y nuevos datos generados manualmente.
- La **normalización de datos** fue fundamental: sin ella, diferencias en escala entre características distorsionarían el cálculo de distancias Euclídeas.

### Limitaciones Identificadas
- K-Means requiere conocer K a priori; en problemas reales, esta elección puede ser subjetiva.
- No es adecuado para clústeres con formas no esféricas o tamaños muy desiguales.
- Sensible a outliers, ya que los centroides son promedios.

> **Comentario Victor:** K-Means es un algoritmo elegante en su simplicidad. Su transparencia algorítmica lo hace ideal para contextos donde la explicabilidad es prioritaria. La interpretación semántica de los clústeres (dar nombre y significado a cada grupo) es el verdadero reto en aplicaciones reales y requiere conocimiento del dominio. En producción, siempre es recomendable combinar el análisis cuantitativo (métricas de silueta, Davies-Bouldin) con el análisis cualitativo (perfiles de centroides) para validar que los grupos tienen sentido de negocio.